In [137]:
from abc import ABC, abstractmethod
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

from cbm import sample_mrf_prec ,GaussianLangevinMechanism, MacroCausalVar, SCBM
from cbm import make_iterable

In [2]:
seed = 0
rs = np.random.RandomState(seed=seed)

In [3]:
# Hardcoded adjacency matrix for testing
A = [[0, 1, 1],
     [0, 0, 1],
     [0, 0, 0]]
A = np.asarray(A)

# Internal dimension of each node
d_micro = 4
# Bottleneck dimension
d_bottleneck = 1
# Hardcode this same internal adjacency for all nodes for now
M = np.ones((4, 4))
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0

In [4]:
def sample_from_simplex(rs, n):
    w = rs.exponential(scale=1.0, size=n)
    return np.expand_dims(w/sum(w), 1)

In [5]:
# Define (or sample at some point, then we can include this on the overall loop) bottleneck functions
bottleneck_fcts = np.full(A.shape[0], None)
w1 = sample_from_simplex(rs=rs, n=d_micro)
fct_1 = lambda x: x @ w1
bottleneck_fcts[0] = fct_1

w2 = sample_from_simplex(rs=rs, n=d_micro)
fct_2 = lambda x: x @ w2
bottleneck_fcts[1] = fct_2

In [6]:
w1

array([[0.21154331],
       [0.33382619],
       [0.24539256],
       [0.20923794]])

In [7]:
# Same thing as above for mu (that maps from bottleneck back to node)
# Make sure that each mu takes as many arguments as it has parents
mus = np.full(A.shape[0], None)
# w3 = np.random.rand(d_bottleneck, d_micro)
mu_1 = lambda  x: x @ np.ones((1, d_micro))
mus[1] = mu_1

# w4 = np.random.rand(d_bottleneck, d_micro)
# w5 = np.random.rand(d_bottleneck, d_micro)
mu_2 = lambda  x1, x2: x1 @ np.ones((1, d_micro)) + x2 @ (2 * np.ones((1, d_micro)))
mus[2] = mu_2

In [8]:
scbm_variables = np.empty(A.shape[0], dtype=object)
# Pretty sure this loop will only work if we have a causal order
for i in range(A.shape[0]):
    # Sample precision matrix for internal mechanism
    P = sample_mrf_prec(dim=d_micro, M=M, rs=rs)

    # Get parents of variables
    parents = A[:, i]
    if np.sum(parents) == 0:  # No parents
        mech = GaussianLangevinMechanism(mu=np.zeros(d_micro), E=np.linalg.inv(P))
        scbm_variables[i] = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech, n=d_micro)
    else:
        parent_idxs = np.nonzero(parents)[0]
        mech = GaussianLangevinMechanism(mu=mus[i], E=np.linalg.inv(P))
        scbm_variables[i] = MacroCausalVar(parents=scbm_variables[parent_idxs],
                                        bottleneck_fcts=bottleneck_fcts[parent_idxs],
                                        mechanism=mech, n=d_micro)

In [9]:
test_scbm = SCBM(variables=scbm_variables, seed=seed)

In [10]:
obs_sample = test_scbm.sample(size=30000)

In [11]:
reg_full = LinearRegression().fit(obs_sample[0], obs_sample[1])
print(F'{reg_full.coef_}')
print(np.mean(reg_full.coef_, axis=0))

[[0.1929978  0.33092159 0.22437611 0.20380742]
 [0.20917927 0.32588153 0.26441379 0.21729291]
 [0.22653784 0.33401391 0.26416803 0.20875905]
 [0.20372337 0.32908646 0.22536596 0.19446313]]
[0.20810957 0.32997587 0.24458097 0.20608063]


In [12]:
w1.T

array([[0.21154331, 0.33382619, 0.24539256, 0.20923794]])

In [13]:
# Reduced Rank Regression
# Get principle axes of Y_hat
Y_hat = obs_sample[0] @ reg_full.coef_
pca = PCA(n_components=1)
pca.fit(Y_hat)
U = pca.components_
print(F'{reg_full.coef_ @ U.T @ U}')

[[0.20558145 0.28373968 0.26811639 0.19915233]
 [0.21901985 0.30228711 0.28564255 0.21217046]
 [0.22262634 0.30726472 0.29034609 0.21566417]
 [0.2056593  0.28384713 0.26821792 0.19922774]]


In [200]:
class BaseRegressor(ABC):
    def __init__(self, d_micro_in, d_micro_out, d_bottleneck, cond=False):
        self.d_micro_in = d_micro_in
        self.d_micro_out = d_micro_out
        self.d_bottleneck = d_bottleneck

        self.cond = cond

    @abstractmethod
    def fit(self, X, Y, X_cond=None):
        raise NotImplementedError

    @abstractmethod
    def get_bottleneck_fct(self):
        """
        This should return a function that can be called to embed samples to the bottleneck space.
        """
        raise NotImplementedError

In [212]:
class LinRegressor(BaseRegressor):
    def __init__(self, d_micro_in, d_micro_out, d_bottleneck, cond=False):
        super().__init__(d_micro_in, d_micro_out, d_bottleneck, cond)

        self.model = LinearRegression()

    def fit(self, X, Y, X_cond=None):
        if self.cond:
            assert X_cond is not None, 'X_cond is None!'
            self.X_dim = X.shape[-1]  # need this later in get_bottleneck_fct

            X_cond = make_iterable(X_cond)
            X_cat = np.concatenate((X, *X_cond), axis=1)
            self.model.fit(X_cat, Y)
        else:
            self.model.fit(X, Y)

    def get_bottleneck_fct(self):
        # using the first d_bottleneck entries...double check if this makes sense
        # TODO: this should be the d_bottleneck lin indep rows!
        if self.cond:
            linear_map = self.model.coef_[:self.d_bottleneck, :self.X_dim].T
        else:
            linear_map = self.model.coef_[:self.d_bottleneck, :].T
        print(f'Linear map: {linear_map}')
        fct = lambda x: x @ linear_map

        return fct

In [16]:
def get_children(A, i):
    """
    Args:
        A: np.array
            Adjacency matrix.
        i: int
            Index of node to get children of.

    Returns:
        children: list[]
            List of indices of children of node at index i.
    """
    return np.nonzero(A[i,:])[0]

In [99]:
def check_open_path(source, target):
    """
    Check if there is an open path between the start and
    end nodes. Probably this is pretty inefficient.
    """
    if source in target.parents:
        return True

    # Check that candidate node has parents
    if target.parents is not None:
        # Only consider candidates that themselves have parents
        candidates = [i for i in target.parents if i.parents is not None]
        for candidate in candidates:
            intermed = check_open_path(source, candidate)
            if intermed:
                return True
    else:
        return False

    return False

In [105]:
P = sample_mrf_prec(dim=d_micro, M=M, rs=rs)
mech = GaussianLangevinMechanism(mu=np.zeros(d_micro), E=np.linalg.inv(P))

X1 = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech, n=d_micro)
X6 = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech, n=d_micro)
X5 = MacroCausalVar(parents=[X1, X6], bottleneck_fcts=None, mechanism=mech, n=d_micro)
X7 = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech, n=d_micro)
X4 = MacroCausalVar(parents=[X7], bottleneck_fcts=None, mechanism=mech, n=d_micro)
X3 = MacroCausalVar(parents=[X4, X5], bottleneck_fcts=None, mechanism=mech, n=d_micro)
X2 = MacroCausalVar(parents=[X1, X3], bottleneck_fcts=None, mechanism=mech, n=d_micro)

In [106]:
print(f'out: {check_open_path(X1, X3)}')

out: True


In [107]:
def front_door_check(source, target):
    """
    Checks if there is a frontdoor path from the start to the end node.
    We assume that end is a child node of start.
    """
    assert source in target.parents, 'End node must be a child of start!'

    # Remove start from list of parents
    parents_sub_start = [i for i in target.parents if i != source]
    if not parents_sub_start:  # List is empty
        return False
    else:
        for parent in parents_sub_start:
            out = check_open_path(source, parent)
            if out:
                return True
        return False

In [108]:
print(f'out: {front_door_check(X1, X2)}')

out: True


In [153]:
# wrapper function to check d-separation
import networkx as nx

def is_d_sep(scm, A, source, target, cond):
    source = set(make_iterable(source))
    target = set(make_iterable(target))
    cond = set(make_iterable(cond))
    G = nx.DiGraph()
    G.add_nodes_from(scm.variables)
    for i in range(A.shape[0]):  # loop over rows of adjacency matrix
        for j in range(A.shape[1]):
            if A[i, j]:
                G.add_edge(scm.variables[i], scm.variables[j])

    return nx.is_d_separator(G, source, target, cond)

In [128]:
from itertools import chain, combinations

def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

In [221]:
# Get how many bottleneck functions (and variables) we need to estimate
estimated_bottleneck_fcts = np.empty(len(test_scbm.variables), dtype=object)

# Choose corresponding regressor depending on mode
mode = 'linear'
if mode == 'linear':
    reg_model = LinRegressor
else:
    pass

# Loop over variables (in causal ordering) and estimate bottleneck function
for i, var in enumerate(test_scbm.variables):
    child_idxs = get_children(A, i)
    # Nodes without children (don't have a bottleneck)
    if child_idxs.size == 0:
        pass
    # Root nodes
    elif var.parents is None:  # Root node
        # Get first child that doesn't have an open frontdoor path
        # (i.e. doesn't require additional conditioning)
        for child_idx in child_idxs:
            if not front_door_check(var, test_scbm.variables[child_idx]):
                break

        regr_target = test_scbm.variables[child_idx]
        regressor = reg_model(d_micro, d_micro, d_bottleneck)

        regressor.fit(var.value, regr_target.value)

        bottleneck_fct = regressor.get_bottleneck_fct()

        estimated_bottleneck_fcts[i] = bottleneck_fct
    else:
        pass
        # Get potential conditioning set (for each child!)
        for child_idx in child_idxs:
            # Loop over powerset of parents, starting with all parents
            for parent_subset in reversed(list(powerset(var.parents))):
                # Empty subset
                if not parent_subset:
                    cond_set = set(var.parents)
                # Check if current node d-separates subset of parents from target.
                # If so, the complement of the parent subset is a valid conditioning set.
                elif is_d_sep(scm=test_scbm,
                              A=A,
                              source=parent_subset,
                              target=test_scbm.variables[child_idx],
                              cond=var):
                    # Get complement
                    pot_cond_set = set(var.parents).difference(parent_subset)
                    # If the set is empty break both loops, we don't need to search further
                    if not pot_cond_set:
                        cond_set = pot_cond_set
                        break
                    try:
                        if len(pot_cond_set) < len(cond_set):
                            cond_set = pot_cond_set
                    except NameError:  # handles case where cond_set is not yet defined
                        cond_set = pot_cond_set
                else:
                    continue
                break
        # If cond_set is empty just to regular regression, otherwise cond. regression!
        if not cond_set:  # Conditioning set is empty
            regr_target = test_scbm.variables[child_idx]
            regressor = reg_model(d_micro, d_micro, d_bottleneck)
            # No conditioning
            regressor.fit(var.value, regr_target.value)

            bottleneck_fct = regressor.get_bottleneck_fct()
            estimated_bottleneck_fcts[i] = bottleneck_fct
        else:
            regr_target = test_scbm.variables[child_idx]
            regressor = reg_model(d_micro_in=d_micro+len(cond_set)*d_bottleneck,
                                  d_micro_out=d_micro,
                                  d_bottleneck=d_bottleneck,
                                  cond=True)
            # Apply (learned) bottleneck functions
            cond_bottleneck_vars = []

            cond_set_idxs = [np.where(test_scbm.variables == cond_var)[0][0] for cond_var in cond_set]
            for cond_set_idx in cond_set_idxs:
                cond_bottleneck_vars.append(estimated_bottleneck_fcts[cond_set_idx](test_scbm.variables[cond_set_idx].value))

            regressor.fit(var.value, regr_target.value, cond_bottleneck_vars)

            bottleneck_fct = regressor.get_bottleneck_fct()
            estimated_bottleneck_fcts[i] = bottleneck_fct


Linear map: [[0.1929978 ]
 [0.33092159]
 [0.22437611]
 [0.20380742]]
Linear map: [[0.25300286]
 [0.48072922]
 [0.26378245]
 [1.01711435]]


In [218]:
aa = np.asarray([1, 2, 3, 4])
print([np.where(aa == [item])[0][0] for item in [2,3]])

[1, 2]


In [113]:
w1

array([[0.21154331],
       [0.33382619],
       [0.24539256],
       [0.20923794]])

In [162]:
w2

array([[0.12557359],
       [0.23657699],
       [0.13115001],
       [0.50669941]])

In [203]:
regressor.model.coef_

array([[0.25273105, 0.48043965, 0.26355777, 1.01680755, 0.20097912,
        0.33567232, 0.24073177, 0.21604986],
       [0.24458144, 0.46504263, 0.25865214, 1.01091584, 0.22176141,
        0.32882395, 0.26204884, 0.21711307],
       [0.25311458, 0.46393679, 0.26719518, 1.01070363, 0.20941119,
        0.33269415, 0.24964917, 0.21803868],
       [0.24669852, 0.47919562, 0.25557613, 1.01628944, 0.23359963,
        0.34415759, 0.25126362, 0.20903554]])

In [115]:
if not get_children(A,2):
    print('no')

no


/var/folders/3t/tsy826vs2hqbz3y19mzxzp1w0000gn/T/ipykernel_7424/2588005462.py:1: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if not get_children(A,2):


In [126]:
np.nonzero(A[0,:])[0]

array([1, 2])